# Part 3. Scalable Processing
## pandas, Polars LazyFrame and PySpark on the verified FDIC panel

**Input:** the immutable Part 2 clean bank–quarter panel (`run_20261114T175314Z`) and its verified Part 1 lineage. **Output:** an independently audited, same-data performance comparison, numerical equivalence evidence, optimized/physical execution plans and observed scaling at two input sizes.

**Protocol:** quarterly aggregation with a date predicate; consecutive-quarter asset growth with a bank-level window; 85-quarter and 170-quarter observed datasets; pandas/Polars/PySpark; one warmup and three measured trials per scenario. Runtime measurements are not cluster measurements. The panel remains unmodified and failure labels are not created here.

**Sequence:** 1. Source/tests → 2. Part 1–2 dependency → 3. Protocol → 4. Controlled experiment → 5. Results/parity → 6. Plans → 7. Figures/scaling → 8. Independent verification/handoff.

### 1. Repository, dependencies and engine-integration tests

The repository revision and pinned packages define the execution environment. Both workloads must produce equivalent results on a synthetic Parquet panel in pandas, Polars and a functioning local Spark JVM before the FDIC benchmark starts. Missing optional dependencies or skipped integration tests are not accepted.

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import sys
import tempfile
import xml.etree.ElementTree as ET

REPOSITORY = "https://github.com/Duongnb58/BFT65B_11230945_Final.git"
REPO_DIR = Path(tempfile.mkdtemp(prefix="bft65b_part3_code_", dir="/content")) / "repository"
subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(REPO_DIR)], check=True)

required_files = (
    "requirements_part3.txt", "run_scalable_processing.py",
    "src/fdic_pipeline/acquisition.py", "src/fdic_pipeline/workflow.py",
    "src/fdic_pipeline/processing_engines.py",
    "src/fdic_pipeline/scalable_processing.py",
    "tests/test_scalable_processing.py", "tests/test_processing_integration.py",
)
missing = [name for name in required_files if not (REPO_DIR / name).is_file()]
assert not missing, f"Missing repository files: {missing}"
COMMIT_SHA = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

# Spark's JVM and Python worker must share the local runtime.
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_PYTHON"] = sys.executable
java_check = subprocess.run(["java", "-version"], text=True, capture_output=True, check=True)
java_line = (java_check.stderr or java_check.stdout).splitlines()[0]
java_match = re.search(r'(?:version|openjdk)\s+"?(\d+)', java_line)
assert java_match is not None and int(java_match.group(1)) >= 17, f"Java 17+ required: {java_line}"

subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                str(REPO_DIR / "requirements_part3.txt")], check=True)
TEST_XML = Path("/content/bft65b_part3_tests.xml")
subprocess.run([
    sys.executable, "-m", "pytest", "-q", "-rs",
    "--junitxml", str(TEST_XML),
    str(REPO_DIR / "tests" / "test_scalable_processing.py"),
    str(REPO_DIR / "tests" / "test_processing_integration.py"),
], cwd=REPO_DIR, check=True)
suites = ET.parse(TEST_XML).getroot().findall(".//testsuite")
assert suites, "Machine-readable pytest results were not produced"
assert sum(int(s.get("skipped", 0)) for s in suites) == 0, "Engine integration tests were skipped"
assert sum(int(s.get("failures", 0)) + int(s.get("errors", 0)) for s in suites) == 0

sys.path.insert(0, str(REPO_DIR / "src"))
print("Part 3 tests: PASSED; skipped: 0")
print("Java:", java_line)
print("Git revision:", COMMIT_SHA)
print("Repository:", REPO_DIR)

### 2. Verified Part 2 panel and Part 1 source lineage

Part 3 consumes the **specified** Part 2 run instead of selecting the most recently created folder. The source contract checks the acquisition identity, Part 2 manifest and validation hash, panel row accounting, 170-quarter inventory and retrospective-exit separation. Section 4 rechecks all 170 Parquet file hashes while staging inputs locally.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
from fdic_pipeline.scalable_processing import source_contract

DRIVE_ROOT = Path("/content/drive/MyDrive/BFT65B_11230945_Final")
ACQUISITION_DIR = DRIVE_ROOT / "fdic_1984Q1_2026Q2"
PART2_RUN_ID = "run_20261114T175314Z"
PART2_DIR = DRIVE_ROOT / "data_quality_panel" / PART2_RUN_ID
SOURCE = source_contract(ACQUISITION_DIR, PART2_DIR)

print("Part 1–2 source integrity: PASSED")
print("Part 2 run:", SOURCE["part2_run_id"])
print("Clean panel rows:", f"{SOURCE['part2_panel_rows']:,}")
print("Verified Parquet partitions:", SOURCE["part2_quarters"])
print("Source schema columns:", SOURCE["panel_fields"])
print("Part 2 manifest SHA-256:", SOURCE["part2_manifest_sha256"])
print("Part 2 validation SHA-256:", SOURCE["part2_validation_sha256"])

### 3. Workload definitions and benchmark identity

The six-column quarterly aggregation applies a report-date filter before grouping. The three-column bank-history workload measures quarter-over-quarter asset growth **only for immediately adjacent reports of the same CERT**. Each input subset contains observed FDIC records, not duplicated synthetic rows. Engine initialization is excluded from the timed interval; one warmup and three measured repetitions share a fresh engine process per scenario.

In [ ]:
from datetime import datetime, timezone
from fdic_pipeline.scalable_processing import SEED, WARMUPS, REPETITIONS, SAMPLES, WORKLOADS, ENGINES

RUN_ID = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
OUTPUT_DIR = DRIVE_ROOT / "scalable_processing" / RUN_ID
LOCAL_DIR = Path("/content") / f"bft65b_part3_stage_{RUN_ID}"
assert not OUTPUT_DIR.exists(), f"Part 3 run already exists: {OUTPUT_DIR}"
assert not LOCAL_DIR.exists(), f"Local staging directory already exists: {LOCAL_DIR}"

print("Run ID:", RUN_ID)
print("Input Part 2 run:", PART2_RUN_ID)
print("Samples:", SAMPLES, "(latest 85 and all 170 quarters)")
print("Workloads:", WORKLOADS)
print("Engines:", ENGINES)
print("Protocol:", WARMUPS, "warmup +", REPETITIONS, "measured per scenario")
print("Total executions:", len(SAMPLES) * len(WORKLOADS) * len(ENGINES) * (WARMUPS + REPETITIONS))
print("Random seed:", SEED)
print("Part 3 evidence:", OUTPUT_DIR)

### 4. Stage verified Parquet input and run controlled benchmarks

Every Part 2 partition is rehashed and copied to the local Colab filesystem. The latest 85-quarter subset uses hard links where supported. Each engine reads these same local Parquet files. A reproducible randomized scenario order is used; numerical parity and row accounting are checked before creating the final evidence manifest.

This section performs the actual benchmark. If an engine fails or outputs disagree, the run remains incomplete and must not be used for the report.

In [ ]:
BENCHMARK_CMD = [
    sys.executable, str(REPO_DIR / "run_scalable_processing.py"), "run",
    "--acquisition-dir", str(ACQUISITION_DIR),
    "--part2-dir", str(PART2_DIR),
    "--output-dir", str(OUTPUT_DIR),
    "--local-dir", str(LOCAL_DIR),
    "--git-commit", COMMIT_SHA,
]
subprocess.run(BENCHMARK_CMD, cwd=REPO_DIR, check=True)

import json
with (OUTPUT_DIR / "part3_validation.json").open(encoding="utf-8") as f:
    VALIDATION = json.load(f)
assert VALIDATION["success"] is True
assert VALIDATION["all_engine_results_match"] is True
assert VALIDATION["measured_trials"] == 36
assert VALIDATION["warmup_trials"] == 12
assert VALIDATION["part2_run_id"] == PART2_RUN_ID
print("Benchmark and numerical equivalence: PASSED")
print("Measured executions:", VALIDATION["measured_trials"])
print("Warmups:", VALIDATION["warmup_trials"])
print("Evidence directory:", OUTPUT_DIR)

### 5. Review numerical equivalence and measured runtime

The independent audit compares complete quarter-level results, not just their row counts. Reported medians exclude warmups. The pandas baseline uses Parquet projection and predicate pushdown, so speed differences are attributable to the measured execution paths rather than a deliberately inefficient input format.

In [ ]:
import csv
from IPython.display import display, Markdown, Image, HTML

def read_csv_evidence(filename):
    with (OUTPUT_DIR / filename).open(newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

def show_table(title, fields, data):
    display(Markdown("**" + title + "**"))
    display(Markdown(" | ".join(fields) + "\n" + " | ".join("---" for _ in fields)
                     + "\n" + "\n".join(" | ".join(str(row[c]) for c in fields)
                                                  for row in data)))

full_results = [r for r in read_csv_evidence("cross_engine_comparisons.csv")
                if r["sample"] == "full"]
show_table("Full-panel latency: median of 3 measured executions",
           ["workload", "input_rows", "pandas_median_s", "polars_median_s",
            "pyspark_median_s", "pandas_over_polars"], full_results)
scaling = read_csv_evidence("scaling.csv")
show_table("Measured input-size and runtime ratios", ["engine", "workload",
           "half_input_rows", "full_input_rows", "row_count_ratio", "time_ratio"], scaling)
with (OUTPUT_DIR / "equivalence.json").open(encoding="utf-8") as f:
    equality = json.load(f)
assert equality["success"] is True
assert len(equality["audited_scenarios"]) == 12
print("Numerical equivalence: all 12 engine/workload/sample scenarios passed")

### 6. Optimized Polars and physical Spark execution plans

Polars plans are recorded **after logical optimization**, before materialization. Spark plans are captured in formatted physical form. Inspect the actual operators rather than inferring optimizer behaviour solely from runtime ratios. A Spark `Exchange` under `local[2]` is a local shuffle, not a measurement of network execution.

In [ ]:
discussion = (OUTPUT_DIR / "execution_plan_discussion.md").read_text(encoding="utf-8")
display(Markdown(discussion))
plan_flags = [r for r in read_csv_evidence("plan_inspection.csv") if r["sample"] == "full"]
show_table("Plan markers in the full-panel scenarios",
           ["engine", "workload", "parquet_scan", "filter_in_plan",
            "aggregation_in_plan", "exchange_or_shuffle", "window_in_plan"], plan_flags)

for engine, workload in (("polars", "quarter_summary"),
                         ("polars", "adjacent_quarter_growth"),
                         ("pyspark", "quarter_summary"),
                         ("pyspark", "adjacent_quarter_growth")):
    plan_file = OUTPUT_DIR / "plans" / f"full_{workload}_{engine}.txt"
    print(f"\n--- {engine} / {workload}: first 30 plan lines ---")
    print("\n".join(plan_file.read_text(encoding="utf-8").splitlines()[:30]))

### 7. Visual comparison and scaling interpretation

Figures show the full-panel median execution times and approximate peak Python-plus-child-process RSS, plus observed scaling between 85 and 170 quarterly partitions. Scaling conclusions must refer to the actual row counts and timings, not assume a 2× row increase or a cluster break-even threshold.

In [ ]:
for title, filename in (
    ("Figure 1. Full-panel runtime by engine", "runtime_comparison.png"),
    ("Figure 2. Observed scaling across two input sizes", "scaling_comparison.png"),
    ("Figure 3. Peak sampled process-tree RSS", "peak_process_rss.png"),
):
    display(Markdown("**" + title + "**"))
    display(Image(filename=str(OUTPUT_DIR / "figures" / filename), width=700))
    display(HTML('<div style="height:20px"></div>'))

### 8. Read-only final validation and Part 4 handoff

The verification command checks that Part 3 still points to the same Part 1–2 source hashes and that every worker result, trial table, plan and figure matches the Part 3 manifest. It does not rerun the benchmark. Part 4 must use the **Part 2 clean panel** for feature construction; Part 3 adds only measured processing evidence, not labels or a new feature table.

In [ ]:
subprocess.run([
    sys.executable, str(REPO_DIR / "run_scalable_processing.py"), "verify",
    "--acquisition-dir", str(ACQUISITION_DIR),
    "--part2-dir", str(PART2_DIR),
    "--output-dir", str(OUTPUT_DIR),
], cwd=REPO_DIR, check=True)
with (OUTPUT_DIR / "part3_validation.json").open(encoding="utf-8") as f:
    final_validation = json.load(f)
assert final_validation["success"] is True
assert final_validation["git_commit"] == COMMIT_SHA
assert final_validation["part2_run_id"] == PART2_RUN_ID
print("Part 3 evidence: PASSED")
print("Part 2 source:", PART2_DIR / "panel")
print("Part 3 run:", RUN_ID)
print("Performance summary:", OUTPUT_DIR / "performance_summary.csv")
print("Execution plans:", OUTPUT_DIR / "plans")
print("Evidence manifest:", OUTPUT_DIR / "part3_manifest.json")
print("Git commit:", COMMIT_SHA)

## 9. Handoff and limitations

The next stage is Part 4 (Feature Pipeline). Continue using the clean Parquet panel from the explicit Part 2 run as the data source. Preserve the Part 3 result directory for the final evaluation of processing efficiency and the single-machine-versus-cluster decision. A local Spark plan does not establish the throughput, cost or latency of a multi-node deployment. The 90-day availability assumption and retrospective exit evidence from Part 2 remain outside the predictor panel.